# Alternative feature selection - 04 final refit and test ranking

This notebook consumes finalist configurations selected by notebook 03 using nested-CV outer-fold evaluation. It refits prescreening and models on the full `X_train/y_train`, scores `X_test`, and saves final top-k rankings for submission and analysis.

No model, feature-set, or ensemble member is selected from test scores. The optional ensemble uses only finalists from notebook 03.

## 0. Setup


In [9]:
from __future__ import annotations

import json
import re
from datetime import UTC, datetime

import pandas as pd

from cost_effective.dataset import find_project_root, load_test_data, load_training_data
from cost_effective.dataset.utils import DEFAULT_MAX_TARGETS
from cost_effective.models.feature_selection_alternative_evaluation import (
    build_rank_mean_ensemble,
    refit_final_configurations,
)
from cost_effective.models.feature_selection_alternative_inner_selection import (
    load_stage_one_tables,
)
from cost_effective.notebook_artifacts import resolve_output_dir
from cost_effective.utils import DEFAULT_SUBMISSION_PREFIX, write_submission_files

project_root = find_project_root()
USE_EXISTING_OUTPUTS = True
FORCE_RERUN = False
stage_one_dir = resolve_output_dir(
    project_root,
    "outputs",
    "feature_selection_alternative",
    "01_fold_plan_and_recipe_space",
    use_existing_outputs=USE_EXISTING_OUTPUTS,
    force_rerun=FORCE_RERUN,
)
stage_three_dir = resolve_output_dir(
    project_root,
    "outputs",
    "feature_selection_alternative",
    "03_fold_evaluation",
    use_existing_outputs=USE_EXISTING_OUTPUTS,
    force_rerun=FORCE_RERUN,
)
outputs = resolve_output_dir(
    project_root,
    "outputs",
    "feature_selection_alternative",
    "04_refit_and_test_ranking",
    use_existing_outputs=USE_EXISTING_OUTPUTS,
    force_rerun=FORCE_RERUN,
)

X_train, y_train = load_training_data(project_root / "data")
X_test = load_test_data(project_root / "data")
stage_one = load_stage_one_tables(stage_one_dir)
prescreen_recipes = stage_one["prescreen_recipes"]
model_specs = stage_one["model_specs"]

X_train.shape, X_test.shape

((5000, 500), (5001, 500))

In [10]:
RANDOM_STATE = 42
TOP_N_TEST = DEFAULT_MAX_TARGETS
TOP_FINALISTS_TO_REFIT = 5
RUN_SIMPLE_ENSEMBLE = True
SUBMISSION_PREFIX = DEFAULT_SUBMISSION_PREFIX

config = {
    "random_state": RANDOM_STATE,
    "top_n_test": TOP_N_TEST,
    "top_finalists_to_refit": TOP_FINALISTS_TO_REFIT,
    "run_simple_ensemble": RUN_SIMPLE_ENSEMBLE,
    "submission_prefix": SUBMISSION_PREFIX,
    "selection_contract": "Finalists come only from notebook 03 alternative outer evaluation.",
}
with (outputs / "stage_config.json").open("w") as file_obj:
    json.dump(config, file_obj, indent=2, default=str)

pd.Series(config)

random_state                                                             42
top_n_test                                                             1000
top_finalists_to_refit                                                    5
run_simple_ensemble                                                    True
submission_prefix                                  pozorski_florek_poltorak
selection_contract        Finalists come only from notebook 03 alternati...
dtype: object

In [11]:
OUTPUT_FILES = {
    "run_log": outputs / "final_refit_run_log.csv",
    "finalists_used": outputs / "notebook03_finalists_used.csv",
    "final_manifest": outputs / "final_refit_manifest.csv",
    "final_predictions": outputs / "final_test_predictions.csv",
    "final_features": outputs / "final_selected_features.csv",
    "ranking_manifest": outputs / "final_ranking_manifest.csv",
    "best_top1000": outputs / "best_final_test_top1000.csv",
    "comparison": outputs / "final_refit_comparison.csv",
    "ensemble_manifest": outputs / "simple_ensemble_manifest.csv",
    "ensemble_top1000": outputs / "simple_ensemble_rank_mean_test_top1000.csv",
    "manifest": outputs / "output_manifest.csv",
    "status": outputs / "stage_status_summary.json",
}


def log_event(stage: str, message: str, **payload) -> None:
    row = {
        "timestamp_utc": datetime.now(UTC).isoformat(),
        "stage": stage,
        "message": message,
        **payload,
    }
    frame = pd.DataFrame([row])
    frame.to_csv(
        OUTPUT_FILES["run_log"],
        mode="a",
        index=False,
        header=not OUTPUT_FILES["run_log"].exists(),
    )
    print(f"[{row['timestamp_utc']}] {stage}: {message} {payload}")


def safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")[:120]


log_event("setup", "initialized notebook 04", output_dir=str(outputs.relative_to(project_root)))

[2026-06-08T14:54:41.146963+00:00] setup: initialized notebook 04 {'output_dir': 'outputs/feature_selection_alternative/04_refit_and_test_ranking'}


## 1. Load finalists selected by notebook 03

This notebook intentionally has no fallback to notebook 02 recipe scores. Finalists must come from `stable_outer_configurations.csv`, which is produced after alternative outer-fold evaluation.


In [13]:
finalists_path = stage_three_dir / "stable_outer_configurations.csv"
if not finalists_path.exists():
    raise FileNotFoundError(
        f"Missing {finalists_path}. Run modeling_feature_selection_alternative_evaluation.ipynb first."
    )

finalists = pd.read_csv(finalists_path)
finalists = (
    finalists.sort_values("finalist_rank").head(TOP_FINALISTS_TO_REFIT).reset_index(drop=True)
)
finalists.to_csv(OUTPUT_FILES["finalists_used"], index=False)

log_event(
    "load_finalists",
    "loaded notebook-03 finalists",
    finalist_rows=len(finalists),
    source=str(finalists_path.relative_to(project_root)),
)
finalists

[2026-06-08T14:54:58.405761+00:00] load_finalists: loaded notebook-03 finalists {'finalist_rows': 5, 'source': 'outputs/feature_selection_alternative/03_fold_evaluation/stable_outer_configurations.csv'}


,recipe_config_id,prescreen_recipe_id,prescreen_name,feature_size,model_spec_id,base_model_family,model_kind,model_params,outer_fold_count,mean_outer_business_score,...,mean_outer_f1_score,mean_outer_roc_auc_score,mean_inner_oof_business_score,mean_inner_selection_rank,selected_feature_count,example_selected_features_text,outer_config_rank,selection_note,stability_note,finalist_rank
0,ps_mean_0007__k_006__extra_trees_small_deep03,ps_mean_0007,single__sparse_gam_spam,6,extra_trees_small_deep03,extra_trees_small,classifier,"{""class_weight"": ""balanced"", ""max_depth"": 5, ""...",5,5450.0,...,0.445211,0.695838,4943.75,7.0,6.0,"var_175,var_214,var_254,var_341,var_379,var_482",1,feature_size <= 6; outer_fold_count == 5,feature_size <= 6; outer_fold_count == 5,1
1,ps_mean_0007__k_006__extra_trees_small_deep08,ps_mean_0007,single__sparse_gam_spam,6,extra_trees_small_deep08,extra_trees_small,classifier,"{""class_weight"": null, ""max_depth"": 5, ""min_sa...",5,5400.0,...,0.443361,0.695104,4940.00,8.0,6.0,"var_175,var_214,var_254,var_341,var_379,var_482",2,feature_size <= 6; outer_fold_count == 5,feature_size <= 6; outer_fold_count == 5,2
2,ps_mean_0007__k_006__extra_trees_small_deep04,ps_mean_0007,single__sparse_gam_spam,6,extra_trees_small_deep04,extra_trees_small,classifier,"{""class_weight"": ""balanced"", ""max_depth"": 6, ""...",5,5355.0,...,0.441444,0.694526,4931.25,11.0,6.0,"var_175,var_214,var_254,var_341,var_379,var_482",3,feature_size <= 6; outer_fold_count == 5,feature_size <= 6; outer_fold_count == 5,3
3,ps_mean_0007__k_006__extra_trees_small_deep02,ps_mean_0007,single__sparse_gam_spam,6,extra_trees_small_deep02,extra_trees_small,classifier,"{""class_weight"": ""balanced"", ""max_depth"": 4, ""...",5,5320.0,...,0.440046,0.693882,4966.25,3.0,6.0,"var_175,var_214,var_254,var_341,var_379,var_482",4,feature_size <= 6; outer_fold_count == 5,feature_size <= 6; outer_fold_count == 5,4
4,ps_mean_0007__k_006__extra_trees_small_deep01,ps_mean_0007,single__sparse_gam_spam,6,extra_trees_small_deep01,extra_trees_small,classifier,"{""class_weight"": ""balanced"", ""max_depth"": 3, ""...",5,5310.0,...,0.439727,0.693065,4972.50,2.0,6.0,"var_175,var_214,var_254,var_341,var_379,var_482",5,feature_size <= 6; outer_fold_count == 5,feature_size <= 6; outer_fold_count == 5,5


In [ ]:
# ── determine submission k from pooled outer OOF (nested-CV estimate at 20% rate) ─
outer_oof_path = stage_three_dir / "outer_oof_scores.csv"
if outer_oof_path.exists():
    outer_oof_df = pd.read_csv(outer_oof_path)
    k_oof_map = (
        outer_oof_df[outer_oof_df["status"] == "ok"]
        .set_index("recipe_config_id")["outer_oof_optimal_k"]
        .to_dict()
    )
    best_finalist_id = finalists.iloc[0]["recipe_config_id"]
    TOP_N_SUBMISSION = int(k_oof_map.get(best_finalist_id, TOP_N_TEST))
    log_event(
        "oof_k",
        "loaded OOF optimal k per finalist from outer_oof_scores.csv",
        k_oof_map={k: int(v) for k, v in k_oof_map.items()},
        best_finalist_id=best_finalist_id,
        top_n_submission=TOP_N_SUBMISSION,
    )
else:
    k_oof_map = {}
    TOP_N_SUBMISSION = TOP_N_TEST
    log_event(
        "oof_k", "outer_oof_scores.csv not found — falling back to TOP_N_TEST", top_n=TOP_N_TEST
    )

print(f"TOP_N_TEST (refit scoring range) : {TOP_N_TEST}")
print(f"TOP_N_SUBMISSION (final cutoff)  : {TOP_N_SUBMISSION}")
print()
if k_oof_map:
    print("k_oof per finalist:")
    for fid, k in sorted(k_oof_map.items(), key=lambda x: x[1], reverse=True):
        print(f"  {k:4d}  {fid}")

## 2. Final refit on all training rows and score test

For each finalist, prescreening is fitted on all `X_train/y_train`, the top feature set is materialized, and the model is fitted on all training rows before scoring `X_test`.


In [15]:
final_manifest, final_test_predictions, final_selected_features = refit_final_configurations(
    X_train,
    y_train,
    X_test,
    finalists,
    prescreen_recipes,
    model_specs,
    random_state=RANDOM_STATE,
    top_n=TOP_N_TEST,
)

final_manifest.to_csv(OUTPUT_FILES["final_manifest"], index=False)
final_test_predictions.to_csv(OUTPUT_FILES["final_predictions"], index=False)
final_selected_features.to_csv(OUTPUT_FILES["final_features"], index=False)

log_event(
    "final_refit",
    "saved final refit outputs",
    manifest_rows=len(final_manifest),
    prediction_rows=len(final_test_predictions),
    selected_feature_rows=len(final_selected_features),
    ok_finalists=int(final_manifest["status"].eq("ok").sum()) if not final_manifest.empty else 0,
)
final_manifest

[final] fitting 1 prescreen methods on all 5000 train rows
[final] ps_mean_0007__k_006__extra_trees_small_deep03 ranked 5001 test rows
[final] ps_mean_0007__k_006__extra_trees_small_deep08 ranked 5001 test rows
[final] ps_mean_0007__k_006__extra_trees_small_deep04 ranked 5001 test rows
[final] ps_mean_0007__k_006__extra_trees_small_deep02 ranked 5001 test rows
[final] ps_mean_0007__k_006__extra_trees_small_deep01 ranked 5001 test rows
[2026-06-08T14:55:38.760648+00:00] final_refit: saved final refit outputs {'manifest_rows': 5, 'prediction_rows': 25005, 'selected_feature_rows': 30, 'ok_finalists': 5}


,recipe_config_id,prescreen_recipe_id,prescreen_name,feature_size,model_spec_id,base_model_family,model_kind,model_params,outer_config_rank,finalist_rank,...,median_outer_business_score,outer_fold_count,mean_outer_optimal_k,selected_features_text,selected_feature_count,test_rows_scored,top_n_saved,status,error,duration_seconds
0,ps_mean_0007__k_006__extra_trees_small_deep03,ps_mean_0007,single__sparse_gam_spam,6,extra_trees_small_deep03,extra_trees_small,classifier,"{""class_weight"": ""balanced"", ""max_depth"": 5, ""...",1,1,...,5525.0,5,199.0,"var_175,var_190,var_214,var_341,var_379,var_482",6,5001,1000,ok,,0.533027
1,ps_mean_0007__k_006__extra_trees_small_deep08,ps_mean_0007,single__sparse_gam_spam,6,extra_trees_small_deep08,extra_trees_small,classifier,"{""class_weight"": null, ""max_depth"": 5, ""min_sa...",2,2,...,5375.0,5,199.8,"var_175,var_190,var_214,var_341,var_379,var_482",6,5001,1000,ok,,0.592527
2,ps_mean_0007__k_006__extra_trees_small_deep04,ps_mean_0007,single__sparse_gam_spam,6,extra_trees_small_deep04,extra_trees_small,classifier,"{""class_weight"": ""balanced"", ""max_depth"": 6, ""...",3,3,...,5350.0,5,198.6,"var_175,var_190,var_214,var_341,var_379,var_482",6,5001,1000,ok,,0.543690
3,ps_mean_0007__k_006__extra_trees_small_deep02,ps_mean_0007,single__sparse_gam_spam,6,extra_trees_small_deep02,extra_trees_small,classifier,"{""class_weight"": ""balanced"", ""max_depth"": 4, ""...",4,4,...,5375.0,5,197.0,"var_175,var_190,var_214,var_341,var_379,var_482",6,5001,1000,ok,,0.531730
4,ps_mean_0007__k_006__extra_trees_small_deep01,ps_mean_0007,single__sparse_gam_spam,6,extra_trees_small_deep01,extra_trees_small,classifier,"{""class_weight"": ""balanced"", ""max_depth"": 3, ""...",5,5,...,5275.0,5,198.6,"var_175,var_190,var_214,var_341,var_379,var_482",6,5001,1000,ok,,0.507569


## 3. Save top-1000 rankings and submission files


In [16]:
ranking_rows = []
ok_manifest = final_manifest.loc[final_manifest["status"].eq("ok")].sort_values("finalist_rank")
if ok_manifest.empty:
    raise RuntimeError("No finalist refit succeeded; inspect final_refit_manifest.csv")

for _, row in ok_manifest.iterrows():
    config_id = row["recipe_config_id"]
    finalist_rank = int(row["finalist_rank"])
    # use per-finalist OOF optimal k; fall back to TOP_N_TEST if not available
    top_n = int(k_oof_map.get(config_id, TOP_N_TEST))
    top = (
        final_test_predictions.loc[final_test_predictions["recipe_config_id"].eq(config_id)]
        .sort_values("rank")
        .head(top_n)
        .copy()
    )
    ranking_path = (
        outputs / f"final_rank_{finalist_rank:02d}__{safe_name(config_id)}_test_top{top_n}.csv"
    )
    top.to_csv(ranking_path, index=False)
    ranking_rows.append({
        "finalist_rank": finalist_rank,
        "recipe_config_id": config_id,
        "ranking_file": str(ranking_path.relative_to(project_root)),
        "top_n_rows": len(top),
        "oof_optimal_k": top_n,
        "selected_feature_count": int(row["selected_feature_count"]),
        "selected_features_text": row["selected_features_text"],
        "mean_outer_business_score": row.get("mean_outer_business_score"),
        "median_outer_business_score": row.get("median_outer_business_score"),
        "outer_fold_count": row.get("outer_fold_count"),
    })

ranking_manifest = pd.DataFrame(ranking_rows)
ranking_manifest.to_csv(OUTPUT_FILES["ranking_manifest"], index=False)

best_row = ok_manifest.iloc[0]
best_config_id = best_row["recipe_config_id"]
best_top = (
    final_test_predictions.loc[final_test_predictions["recipe_config_id"].eq(best_config_id)]
    .sort_values("rank")
    .head(TOP_N_SUBMISSION)
    .copy()
)
best_top.to_csv(OUTPUT_FILES["best_top1000"], index=False)
best_features = (
    final_selected_features.loc[final_selected_features["recipe_config_id"].eq(best_config_id)]
    .sort_values("feature_order")["feature"]
    .tolist()
)
obs_path, vars_path = write_submission_files(
    outputs,
    SUBMISSION_PREFIX,
    best_top["sample_index"].to_numpy(),
    best_features,
)

log_event(
    "rankings",
    "saved finalist rankings and best submission files",
    ranking_files=len(ranking_manifest),
    best_config_id=best_config_id,
    top_n_submission=TOP_N_SUBMISSION,
    obs_file=str(obs_path.relative_to(project_root)),
    vars_file=str(vars_path.relative_to(project_root)),
)
ranking_manifest

[2026-06-08T14:55:55.094899+00:00] rankings: saved finalist rankings and best submission files {'ranking_files': 5, 'best_config_id': 'ps_mean_0007__k_006__extra_trees_small_deep03', 'top_n_submission': 997, 'obs_file': 'outputs/feature_selection_alternative/04_refit_and_test_ranking/pozorski_florek_poltorak_obs.txt', 'vars_file': 'outputs/feature_selection_alternative/04_refit_and_test_ranking/pozorski_florek_poltorak_vars.txt'}


,finalist_rank,recipe_config_id,ranking_file,top_n_rows,oof_optimal_k,selected_feature_count,selected_features_text,mean_outer_business_score,median_outer_business_score,outer_fold_count
0,1,ps_mean_0007__k_006__extra_trees_small_deep03,outputs/feature_selection_alternative/04_refit...,997,997,6,"var_175,var_190,var_214,var_341,var_379,var_482",5450.0,5525.0,5
1,2,ps_mean_0007__k_006__extra_trees_small_deep08,outputs/feature_selection_alternative/04_refit...,999,999,6,"var_175,var_190,var_214,var_341,var_379,var_482",5400.0,5375.0,5
2,3,ps_mean_0007__k_006__extra_trees_small_deep04,outputs/feature_selection_alternative/04_refit...,1000,1000,6,"var_175,var_190,var_214,var_341,var_379,var_482",5355.0,5350.0,5
3,4,ps_mean_0007__k_006__extra_trees_small_deep02,outputs/feature_selection_alternative/04_refit...,1000,1000,6,"var_175,var_190,var_214,var_341,var_379,var_482",5320.0,5375.0,5
4,5,ps_mean_0007__k_006__extra_trees_small_deep01,outputs/feature_selection_alternative/04_refit...,999,999,6,"var_175,var_190,var_214,var_341,var_379,var_482",5310.0,5275.0,5


## 4. Optional simple ensemble of notebook-03 finalists

This ensemble is only built from finalists already selected by alternative outer evaluation. It is saved for analysis; the primary best submission above remains the top notebook-03 finalist refit.


In [17]:
if RUN_SIMPLE_ENSEMBLE:
    ensemble_manifest, ensemble_top1000 = build_rank_mean_ensemble(
        final_test_predictions,
        final_manifest,
        top_n=TOP_N_SUBMISSION,
        ensemble_name="notebook03_finalist_rank_mean",
    )
    if not ensemble_manifest.empty:
        ensemble_manifest.to_csv(OUTPUT_FILES["ensemble_manifest"], index=False)
        ensemble_top1000.to_csv(OUTPUT_FILES["ensemble_top1000"], index=False)
        log_event(
            "ensemble",
            "saved simple rank-mean ensemble from notebook-03 finalists",
            member_count=int(ensemble_manifest.iloc[0]["member_count"]),
            top_n_rows=len(ensemble_top1000),
            top_n_submission=TOP_N_SUBMISSION,
        )
    else:
        log_event(
            "ensemble",
            "skipped ensemble because fewer than two successful finalists were available",
        )
else:
    ensemble_manifest = pd.DataFrame()
    ensemble_top1000 = pd.DataFrame()
    log_event("ensemble", "disabled by config")

ensemble_manifest

[2026-06-08T14:56:00.943807+00:00] ensemble: saved simple rank-mean ensemble from notebook-03 finalists {'member_count': 5, 'top_n_rows': 997, 'top_n_submission': 997}


,ensemble_name,member_count,member_recipe_config_ids,top_n_saved,selection_rule
0,notebook03_finalist_rank_mean,5,"ps_mean_0007__k_006__extra_trees_small_deep01,...",997,members selected by honest outer evaluation in...


## 5. Comparison frames and output manifest


In [18]:
outer_config_summary_path = stage_three_dir / "outer_config_summary.csv"
outer_config_summary = (
    pd.read_csv(outer_config_summary_path) if outer_config_summary_path.exists() else pd.DataFrame()
)
if not outer_config_summary.empty:
    final_refit_comparison = final_manifest.merge(
        outer_config_summary,
        on="recipe_config_id",
        how="left",
        suffixes=("_final", "_outer"),
    )
else:
    final_refit_comparison = final_manifest.copy()
final_refit_comparison.to_csv(OUTPUT_FILES["comparison"], index=False)

status_summary = {
    "finalist_rows_loaded": len(finalists),
    "successful_final_refits": int(final_manifest["status"].eq("ok").sum())
    if not final_manifest.empty
    else 0,
    "test_prediction_rows": len(final_test_predictions),
    "best_recipe_config_id": str(best_config_id),
    "top_n_test": TOP_N_TEST,
    "top_n_submission": TOP_N_SUBMISSION,
    "best_submission_rows": len(best_top),
    "best_feature_count": len(best_features),
    "oof_k_used": TOP_N_SUBMISSION != TOP_N_TEST,
    "obs_file": str(obs_path.relative_to(project_root)),
    "vars_file": str(vars_path.relative_to(project_root)),
    "ensemble_written": bool(not ensemble_manifest.empty),
}
with OUTPUT_FILES["status"].open("w") as file_obj:
    json.dump(status_summary, file_obj, indent=2, default=str)

output_manifest = pd.DataFrame([
    {"file": "stage_config.json", "meaning": "Notebook-04 execution config."},
    {"file": "final_refit_run_log.csv", "meaning": "Timestamped notebook-04 log."},
    {"file": "notebook03_finalists_used.csv", "meaning": "Finalists loaded from notebook 03."},
    {"file": "final_refit_manifest.csv", "meaning": "One row per final refit configuration."},
    {
        "file": "final_test_predictions.csv",
        "meaning": "All test scores for all successful finalists.",
    },
    {
        "file": "final_selected_features.csv",
        "meaning": "Features selected after final full-train prescreen refit.",
    },
    {
        "file": "final_ranking_manifest.csv",
        "meaning": "Per-finalist top-k ranking file index (k = OOF optimal k per config).",
    },
    {
        "file": "best_final_test_top1000.csv",
        "meaning": f"Top-{TOP_N_SUBMISSION} for the highest-ranked notebook-03 finalist (OOF optimal k).",
    },
    {
        "file": f"{SUBMISSION_PREFIX}_obs.txt",
        "meaning": f"Submission observation indices — top-{TOP_N_SUBMISSION} by OOF optimal k.",
    },
    {
        "file": f"{SUBMISSION_PREFIX}_vars.txt",
        "meaning": "Submission variable indices for the best finalist.",
    },
    {
        "file": "simple_ensemble_manifest.csv",
        "meaning": "Optional rank-mean ensemble manifest, if available.",
    },
    {
        "file": "simple_ensemble_rank_mean_test_top1000.csv",
        "meaning": f"Optional rank-mean ensemble top-{TOP_N_SUBMISSION}, if available.",
    },
    {
        "file": "final_refit_comparison.csv",
        "meaning": "Final refit manifest joined to notebook-03 outer summary.",
    },
    {"file": "stage_status_summary.json", "meaning": "Compact notebook-04 status summary."},
])
output_manifest.to_csv(OUTPUT_FILES["manifest"], index=False)

pd.Series(status_summary)

finalist_rows_loaded                                                       5
successful_final_refits                                                    5
test_prediction_rows                                                   25005
best_recipe_config_id          ps_mean_0007__k_006__extra_trees_small_deep03
top_n_test                                                              1000
top_n_submission                                                         997
best_submission_rows                                                     997
best_feature_count                                                         6
oof_k_used                                                              True
obs_file                   outputs/feature_selection_alternative/04_refit...
vars_file                  outputs/feature_selection_alternative/04_refit...
ensemble_written                                                        True
dtype: object